In [1]:
import pandas as pd
from pathlib import Path

from tgbs_rs.plotting.table_prep import *

#### Read raw Data Tables

In [6]:
data_dir = Path("/Users/chris/Desktop/Projects/TGBS_Base/outputs/tables")

csv_paths = {
    "hls_annual": data_dir / "tgbs_hls_annual_composite_all_bands_2014_2025_full_raw.csv",
    "hls_monthly": data_dir / "tgbs_hls_monthly_composite_all_bands_2014_2025_full_raw.csv",
    "s2_annual": data_dir / "tgbs_s2_annual_composite_all_bands_2019_2025_full_raw.csv",
    "s2_monthly": data_dir / "tgbs_s2_monthly_composite_all_bands_2019_2025_full_raw.csv",
}

productivity_cols = ["NIRv", "EVI", "NDVI"]

In [7]:
hls_annual_df = pd.read_csv(csv_paths["hls_annual"])
hls_monthly_df = pd.read_csv(csv_paths["hls_monthly"])
s2_annual_df = pd.read_csv(csv_paths["s2_annual"])
s2_monthly_df = pd.read_csv(csv_paths["s2_monthly"])

#### Prepare Annual Tables
    - remove structurally irrelevant temporal columns
    - add baseline/current period labels
    - add missing-value flags
    - sort for reproducibility

In [8]:
hls_annual_prepped = prepare_composite_table(
    hls_annual_df,
    temporal_scale="annual",
    value_columns=productivity_cols,
    drop_incomplete_rows=False,
)

In [17]:
s2_annual_prepped = prepare_composite_table(
    df=s2_annual_df,
    temporal_scale="annual",
    value_columns=productivity_cols,
    drop_incomplete_rows=False,
)

In [18]:
s2_annual_prepped.to_csv("/Users/chris/Desktop/Projects/TGBS_Base/outputs/tables/tgbs_s2_annual_composite_all_bands_2018_2025_full_clean.csv")

In [13]:
hls_annual_prepped = keep_core_columns(
    df=hls_annual_prepped,
    value_columns=productivity_cols,
)

##### Prepare Monthly Tables
    - remove structurally irrelevant temporal columns
    - add baseline/current period labels
    - add missing-value flags
    - sort for reproducibility

In [15]:
hls_monthly_prepped = prepare_composite_table(
    df=hls_monthly_df,
    temporal_scale="monthly",
    value_columns=productivity_cols,
    drop_incomplete_rows=False,
)

In [19]:
s2_monthly_prepped = prepare_composite_table(
    df=s2_monthly_df,
    temporal_scale="monthly",
    value_columns=productivity_cols,
    drop_incomplete_rows=False,
)

##### Expand HLS monthly table onto the full expected site-year-month grid
    - makes missing site-month rows explicit

In [17]:
hls_monthly_prepped = merge_monthly_to_full_grid(
    monthly_df=hls_monthly_prepped,
    start_year=2014,
    end_year=2025,
)

In [20]:
s2_monthly_prepped = merge_monthly_to_full_grid(
    monthly_df=s2_monthly_prepped,
    start_year=2019,
    end_year=2025,
)

##### Re-add period and season labels after grid completion
    - period is useful for baseline/current summaries
    - season is needed for wet/dry aggregation

In [19]:
hls_monthly_prepped = add_period_label(
    df=hls_monthly_prepped,
    baseline_start=2014,
    baseline_end=2017,
    current_start=2018,
    current_end=2025,
)

hls_monthly_prepped = add_season_label(
    df=hls_monthly_prepped,
    wet_months=[3, 4, 5],
    dry_months=[7, 8, 9, 10],
)

hls_monthly_prepped = add_valid_value_flag(
    df=hls_monthly_prepped,
    value_columns=productivity_cols,
)

hls_monthly_prepped = flag_rows_with_missing_values(
    df=hls_monthly_prepped,
    value_columns=productivity_cols,
)

hls_monthly_prepped = sort_site_time(hls_monthly_prepped)

In [21]:
s2_monthly_prepped = add_period_label(
    df=s2_monthly_prepped,
    baseline_start=2014,
    baseline_end=2017,
    current_start=2018,
    current_end=2025,
)

s2_monthly_prepped = add_season_label(
    df=s2_monthly_prepped,
    wet_months=[3, 4, 5],
    dry_months=[7, 8, 9, 10],
)

s2_monthly_prepped = add_valid_value_flag(
    df=s2_monthly_prepped,
    value_columns=productivity_cols,
)

s2_monthly_prepped = flag_rows_with_missing_values(
    df=s2_monthly_prepped,
    value_columns=productivity_cols,
)

s2_monthly_prepped = sort_site_time(s2_monthly_prepped)


In [22]:
s2_monthly_prepped.to_csv("/Users/chris/Desktop/Projects/TGBS_Base/outputs/tables/tgbs_s2_monthly_composite_all_bands_2018_2025_full_clean.csv")

##### Generate Seasonal Tables

In [22]:
hls_seasonal_df = aggregate_monthly_to_seasonal_with_support(
    df=hls_monthly_prepped,
    value_columns=productivity_cols,
    min_valid_months_by_season={"wet": 2, "dry": 3},
)

In [23]:
s2_seasonal_df = aggregate_monthly_to_seasonal_with_support(
    df=s2_monthly_prepped,
    value_columns=productivity_cols,
    min_valid_months_by_season={"wet": 2, "dry": 3},
)

In [24]:
s2_seasonal_df.to_csv("/Users/chris/Desktop/Projects/TGBS_Base/outputs/tables/tgbs_s2_seasonal_composite_all_bands_2018_2025_full_clean.csv")